# Session 1 — Spectroscopy with VeloxChem

**eChem School — Marseille 2026**

This session introduces three families of molecular spectroscopy:

1. **UV/vis absorption** — TDDFT eigenvalue approach and the complex polarization propagator (CPP)
2. **Electronic circular dichroism (ECD)** — rotatory strengths (TDDFT) and the extinction coefficient (CPP)

# Part 1 — UV/vis Absorption

## 1.1 TDDFT eigenvalue approach

In the eigenvalue (linear-response) approach, UV/vis spectra are obtained by solving
the **generalized TDDFT eigenvalue equation** for the $N$ lowest excitation energies
$\omega_{n0}$ and their associated transition moments.

The **oscillator strength** for transition $0 \to n$ is

$$
f_{n0} = \frac{2 m_\mathrm{e} \omega_{n0}}{3\hbar e^2}
\sum_{\alpha = x,y,z} |\langle 0 | \hat{\mu}_\alpha | n \rangle |^2
$$

The **linear absorption cross section** is then

$$
\sigma(\omega) =
\frac{2\pi^2 e^2 \omega}{(4\pi\varepsilon_0) m_\mathrm{e} c}
\sum_{n > 0} f(\omega;\, \omega_{n0}, \gamma)\,
\frac{f_{n0}}{\omega_{n0}}
$$

where $f$ is a Cauchy (Lorentzian) line-shape function and $\gamma$ sets the broadening.

In [ ]:
import veloxchem as vlx

# TODO: Create the molecule object for paranitroaniline
molecule = 
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "cam-b3lyp"
scf_results = scf_drv.compute(molecule, basis)

rsp_drv = vlx.LinearResponseEigenSolver()
rsp_drv.nstates = 5
rsp_results = rsp_drv.compute(molecule, basis, scf_results)

In [ ]:
# TODO: use the plot_uv_vis function to plot the UV-Vis spectrum


## 1.2 Complex polarization propagator (CPP)

Instead of discrete excitation energies, the **CPP approach** evaluates the imaginary part
of the frequency-dependent polarizability directly on a user-defined frequency grid:

$$
\sigma(\omega) =
\frac{\omega}{\varepsilon_0 c}\,
\mathrm{Im}\left\{\bar{\alpha}(-\omega;\omega)\right\}
$$

where $\bar{\alpha} = \tfrac{1}{3}(\alpha_{xx}+\alpha_{yy}+\alpha_{zz})$ is the
isotropic polarizability evaluated with a finite damping $\hbar\gamma = 0.124\,\mathrm{eV}$.

The result is a directly broadened spectrum that spans an arbitrary frequency window —
useful when many states contribute or when a continuum region must be described.

In [ ]:
import veloxchem as vlx
import numpy as np

molecule = vlx.Molecule.read_name("ethylene")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "cam-b3lyp"
scf_results = scf_drv.compute(molecule, basis)

cpp_drv = vlx.ComplexResponse()

# TODO: Specify the frequencies and the property to compute for the complex response calculation
# The frequencies should be express in atomic units (1 a.u. of energy = 27.2114 eV) and should cover the range of interest for the UV-Vis spectrum (e.g., from 0.2 to 0.35 a.u. with a step of 0.0025 a.u.)
cpp_drv.frequencies = 
cpp_drv.property = 

cpp_results = cpp_drv.compute(molecule, basis, scf_results)

In [ ]:
cpp_drv.plot(cpp_results)

# Part 2 — Electronic Circular Dichroism (ECD)

ECD measures the **difference in absorption of left and right circularly polarized light**
and is non-zero only for chiral molecules.  The ECD spectrum is expressed as the
anisotropy of the decadic molar extinction coefficient

$$
\Delta\epsilon(\omega) =
\frac{16\pi N_\mathrm{A}}{\ln(10)(4\pi\varepsilon_0)c^2}
\frac{\pi}{3\hbar}
\sum_{n>0} f(\omega;\,\omega_{n0},\gamma)\,\omega_{n0}\, R_{n0}
$$

where the **rotatory strength** is evaluated in the velocity gauge (gauge-origin independent):

$$
R_{n0} =
\sum_\alpha
\frac{-e}{m_\mathrm{e}\omega_{n0}}
\langle 0|\hat{p}_\alpha|n\rangle
\langle n|\hat{m}_\alpha|0\rangle
$$

We use *(S)-methyloxirane* as a chiral test molecule.

In [ ]:
import veloxchem as vlx

methyloxirane_xyz = """10

O             -1.009866880244         1.299407071912         1.951947754409
C              0.031038818173         2.001294498224         1.244371321259
C              0.339506903623         0.795807201271         2.029917688849
C              0.600316751832        -0.544462572436         1.394186918859
H             -0.026285836651         1.930258644799         0.154459855148
H              0.271804379347         2.990968281608         1.641505800108
H              0.794935917667         0.948100885444         3.014899839267
H              0.113110324011        -0.610670447580         0.412641743927
H              1.681576973773        -0.701096452623         1.264343536564
H              0.213803265264        -1.354101659046         2.029564064183"""

# TODO: Create the molecule object for methyloxirane and visualize it


## 2.1 TDDFT rotatory strengths

In [ ]:
basis = vlx.MolecularBasis.read(molecule, "aug-cc-pvdz")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_results = scf_drv.compute(molecule, basis)

rsp_drv = vlx.LinearResponseEigenSolver()
rsp_drv.nstates = 8
rsp_results = rsp_drv.compute(molecule, basis, scf_results)

In [ ]:
# TODO: use the plot_ecd function to plot the ECD spectrum


## 2.2 CPP extinction coefficient

The CPP approach yields $\Delta\epsilon(\omega)$ directly from the mixed
electric–magnetic dipole response:

$$
\Delta\epsilon(\omega) =
\frac{16\pi N_\mathrm{A}\omega^2}{\ln(10)(4\pi\varepsilon_0)c^2}\,\beta(\omega),
\qquad
\beta(\omega) = -\frac{1}{3\omega}(G_{xx}+G_{yy}+G_{zz})
$$

where $G_{\alpha\beta} = -\tfrac{e}{\omega m_e}\,\mathrm{Im}\langle\langle\hat{p}_\alpha;\hat{m}_\beta\rangle\rangle_\omega^\gamma$
is evaluated in the velocity gauge.

In [ ]:
import numpy as np
import veloxchem as vlx

# reuse molecule / basis / scf_results from above
cpp_ecd = vlx.ComplexResponse()

# TODO: Specify the property
cpp_ecd.property = 
cpp_ecd.frequencies = np.arange(0.207, 0.325, 0.0025)


cpp_results_ecd = cpp_ecd.compute(molecule, basis, scf_results)

In [ ]:
cpp_ecd.plot(cpp_results_ecd, x_unit='nm')

# Part 3 — Vibrational Spectroscopy

VeloxChem computes the **molecular Hessian** and transforms it to mass-weighted
normal coordinates.  From these normal modes three vibrational spectra can be obtained:

| Spectrum | Physical quantity | Selection rule |
|----------|-------------------|----------------|
| **IR** | Dipole-moment derivative | $\partial\mu/\partial Q_k \ne 0$ |
| **Raman** | Polarizability derivative | $\partial\alpha/\partial Q_k \ne 0$ |

> **Note:** It is recommended to optimize the geometry before vibrational calculations.

We use **formaldehyde** (H₂CO) as a small, well-characterized test molecule.

## 3.1 Infrared (IR)

IR intensities are proportional to the squared dipole-moment derivative along each
normal mode $Q_k$:

$$
I_k^\mathrm{IR} \propto \left|\frac{\partial\boldsymbol{\mu}}{\partial Q_k}\right|^2
$$

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("formaldehyde")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_results = scf_drv.compute(molecule, basis)

vib_drv = vlx.VibrationalAnalysis(scf_drv)
vib_drv.do_ir = True

vib_results = vib_drv.compute(molecule, basis)

In [ ]:
vib_drv.plot_ir(vib_results)

In [ ]:
# TODO: use the animate function to visualize the vibrational modes


## 3.2 Raman

Raman intensities are proportional to derivatives of the **polarizability tensor**:

$$
I_k^\mathrm{Raman} \propto
45\left(\bar{\alpha}'_k\right)^2 + 7\left(\gamma'_k\right)^2
$$

where $\bar{\alpha}'_k$ and $\gamma'_k$ are the isotropic and anisotropic invariants of
$\partial\boldsymbol{\alpha}/\partial Q_k$. Raman and IR are complementary:
modes inactive in one are often active in the other.

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("formaldehyde")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_results = scf_drv.compute(molecule, basis)

vib_drv = vlx.VibrationalAnalysis(scf_drv)
vib_drv.do_ir = False
vib_drv.do_raman = True

vib_results = vib_drv.compute(molecule, basis)

In [ ]:
vib_drv.plot_raman(vib_results)

# Part 4 — X-Ray spectroscopies

## 4.1 XPS

In [ ]:

molecule = vlx.Molecule.read_xyz_string("""13
ESCA b3lyp/def2-svp optimized geometry
C              1.326024532622         0.471089167011        -1.154937296389
C              0.484399509715         0.908494386259         0.037569541020
C              0.270494167717        -0.188493907246         1.053914467110
O              0.701245126416        -1.308212251798         0.971080711541
C             -0.582260522270         0.210893702941         2.291730106998
F             -0.731967860619        -0.799813015071         3.135965340271
F             -1.800383044157         0.625073687043         1.897202571864
F              0.003899498844         1.231896946553         2.943599980006
H              1.453639490166         1.302028888463        -1.864264306255
H              0.853135331881        -0.368249103011        -1.686230702784
H              2.322713498820         0.134038471332        -0.833394707486
H              0.936346512410         1.764030447212         0.571699332428
H             -0.514670521097         1.267150587304        -0.270103071648""")

basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_results = scf_drv.compute(molecule, basis)

xps_drv = vlx.XPSDriver()

xps_results_c = xps_drv.compute(molecule, basis, scf_drv, element='C')

In [ ]:
xps_drv.plot_spectrum(xps_results_c)

## 4.2 NEXAFS

In [ ]:
import veloxchem as vlx
import numpy as np

molecule = vlx.Molecule.read_xyz_string("""13
ESCA b3lyp/def2-svp optimized geometry
C              1.326024532622         0.471089167011        -1.154937296389
C              0.484399509715         0.908494386259         0.037569541020
C              0.270494167717        -0.188493907246         1.053914467110
O              0.701245126416        -1.308212251798         0.971080711541
C             -0.582260522270         0.210893702941         2.291730106998
F             -0.731967860619        -0.799813015071         3.135965340271
F             -1.800383044157         0.625073687043         1.897202571864
F              0.003899498844         1.231896946553         2.943599980006
H              1.453639490166         1.302028888463        -1.864264306255
H              0.853135331881        -0.368249103011        -1.686230702784
H              2.322713498820         0.134038471332        -0.833394707486
H              0.936346512410         1.764030447212         0.571699332428
H             -0.514670521097         1.267150587304        -0.270103071648""")

basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "cam-b3lyp-100"
scf_results = scf_drv.compute(molecule, basis)

cpp_drv = vlx.ComplexResponse()
cpp_drv.frequencies = np.arange(10.1, 10.4, 0.0025)
cpp_drv.property = "absorption"

cpp_results = cpp_drv.compute(molecule, basis, scf_results)

In [ ]:
cpp_drv.plot(cpp_results, x_unit='eV')

## Summary

| Spectroscopy      | VeloxChem driver         | Key setting                                 |
|------------------|-------------------------|---------------------------------------------|
| UV/vis (TDDFT)   | `LinearResponseEigenSolver` | `nstates`                                 |
| UV/vis (CPP)     | `ComplexResponse`       | `frequencies`, `property="absorption"`     |
| ECD (TDDFT)      | `LinearResponseEigenSolver` | `plot_ecd()`                              |
| ECD (CPP)        | `ComplexResponse`       | `property="ecd"`                          |
| IR               | `VibrationalAnalysis`   | `do_ir=True`                               |
| Raman            | `VibrationalAnalysis`   | `do_raman=True`                            |
| XPS              | `XpsDriver`             | `element` |
| NEXAFS (CPP)           | `NexafsDriver`          | `frequencies`            |
